In [78]:
import pandas as pd
import networkx as nx
import numpy as np
from itertools import combinations
from collections import Counter
from scipy.sparse import csr_matrix, triu


In [51]:
#Reading in the dataset file
file_path = "/Users/june/Downloads/user_games.parquet"

user_games = pd.read_parquet(file_path)

print(type(user_games))

#Cleaning up the dataset file
user_games = user_games[["steamid", "game_name"]].copy()
user_games = user_games.dropna()
user_games = user_games.drop_duplicates(subset = ["steamid", "game_name"])
user_games = user_games.rename(columns={"steamid":"user_id", "game_name":"game"})


<class 'pandas.core.frame.DataFrame'>


In [52]:
user_games["game_id"], unique_games = pd.factorize(user_games["game"])
user_games["game_id"] = user_games["game_id"] + 1

In [53]:
print(user_games[["user_id", "game_id", "game"]].head(20))

              user_id  game_id                                           game
0   76561197960293516        1                                 Counter-Strike
1   76561197960293516        2                          Team Fortress Classic
2   76561197960293516        3                                  Day of Defeat
3   76561197960293516        4                             Deathmatch Classic
4   76561197960293516        5                      Half-Life: Opposing Force
5   76561197960293516        6                                       Ricochet
6   76561197960293516        7                                      Half-Life
7   76561197960293516        8                 Counter-Strike: Condition Zero
8   76561197960293516        9  Counter-Strike: Condition Zero Deleted Scenes
9   76561197960293516       10                          Half-Life: Blue Shift
10  76561197960293516       11                                    Half-Life 2
11  76561197960293516       12                         Counter-S

In [60]:
#Creating a function to look up games with ID quickly
game_lookup = (user_games[["game_id", "game"]].drop_duplicates().sort_values("game_id").reset_index(drop=True))
id_to_game = dict(zip(game_lookup["game_id"], game_lookup["game"]))

#Testing the function
print(id_to_game[100])

Zuma Deluxe


In [64]:
#Creating the networkx graph and populating it with nodes
g = nx.Graph()

for game_id in user_games["game_id"]:
    g.add_node(game_id, game_name = id_to_game.get(game_id))


In [68]:
#Making edges, failed due to time constraint. Too many edges.
"""
for user_id, group in user_games.groupby("user_id"):
    games = group["game_id"].unique()
    for game1, game2 in combinations(games,2):
        if g.has_edge(game1, game2):
            g[game1][game2]["weight"] += 1
        else:
            g.add_edge(game1, game2, weight = 1)
"""


KeyboardInterrupt: 

In [ ]:
#Creating edge matrix to identify edges between games, and make the process faster
user_codes, unique_users = pd.factorize(user_games["user_id"])
game_codes, unique_games = pd.factorize(user_games["game_id"])

user_game_matrix = csr_matrix((np.ones(len(user_games), dtype=np.int32), (user_codes, game_codes)), shape = (len(unique_users), len(unique_games)))

shared = user_game_matrix.T@user_game_matrix
shared.setdiag(0)
shared.eliminate_zeros()


In [79]:
#Since there is a pair listed for both ways, dividing the edge matrix into two
shared_upper = triu(
    shared,
    k=1,
    format = "coo"
)

In [80]:
#Print total number of edges
print("Edges", shared_upper.nnz)

Edges 1120568724


In [83]:
#Checking the edge matrix in order to sparse it
print("Min", shared.data.min())
print("Median", np.median(shared.data))
print("90th percentile", np.percentile(shared.data, 90))
print("90th percentile", np.percentile(shared.data, 95))
print("90th percentile", np.percentile(shared.data, 99))
print("Max", shared.data.max())

Min 1
Median 22.0
90th percentile 251.0
90th percentile 420.0
90th percentile 619.0
Max 785


In [86]:
#Filtering the edges
min_shared_users = 500
shared_filtered = shared.copy()
shared_filtered.data[
    shared_filtered.data < min_shared_users
] = 0
shared_filtered.eliminate_zeros()

shared_upper = triu(
    shared_filtered,
    k=1,
    format = "coo"
)

print("Edges after filtering:", shared_upper.nnz)

Edges after filtering: 35481241


In [87]:
#Finding threshold for filtering
thresholds = [550, 600, 650, 700, 725, 750, 775]

for threshold in thresholds:
    count = (shared_upper.data >= threshold).sum()
    print(f"{threshold}+: {count:,} edges")

550+: 23,783,972 edges
600+: 14,276,658 edges
650+: 7,156,454 edges
700+: 2,464,744 edges
725+: 1,087,025 edges
750+: 289,667 edges
775+: 10,236 edges


In [97]:
#Filtering the edges again, then creating dataframe of edges
min_shared_users = 750

mask = shared_upper.data >= min_shared_users

edges_df = pd.DataFrame({
    "game1": unique_games[shared_upper.row[mask]],
    "game2": unique_games[shared_upper.col[mask]],
    "weight": shared_upper.data[mask]
})

print(len(edges_df))


289667


In [100]:
#Had to create new graph since some of the games are not in the result anymore
g = nx.Graph()

g.add_weighted_edges_from(
    edges_df[["game1", "game2", "weight"]]
    .itertuples(index=False, name=None)
)

In [102]:
#Checking if the edges are all added
print("Nodes", g.number_of_nodes())
print("Edges", g.number_of_edges())

Nodes 1171
Edges 289667


In [103]:
#Identifying importan nodes
#1. Degree Centrality
degree_centrality = nx.degree_centrality(g)
# 2. Weighted degree
weighted_degree = dict(g.degree(weight="weight"))

# 3. Betweenness centrality
for u, v, data in g.edges(data=True):
    data["distance"] = 1 / data["weight"]

betweenness_centrality = nx.betweenness_centrality(
    g,
    weight="distance"
)

In [107]:
#Putting all results to a single dataframe for clean visual
results = pd.DataFrame({
    "game_id": list(g.nodes())
})
results["game"] = results["game_id"].map(id_to_game)
results["degree_centrality"] = (
    results["game_id"].map(degree_centrality)
)

results["weighted_degree"] = (
    results["game_id"].map(weighted_degree)
)

results["betweenness_centrality"] = (
    results["game_id"].map(betweenness_centrality)
)

In [108]:
#Top 10 Games by degree centrality
top_degree = (
    results
    .sort_values(
        "degree_centrality",
        ascending=False
    )
    .head(10)
)

print(
    top_degree[
        ["game", "degree_centrality"]
    ]
)

                        game  degree_centrality
162                Syberia 2           1.000000
10                    Portal           0.984615
161                  Syberia           0.979487
277         Zombie Driver HD           0.970940
283                 POSTAL 2           0.970940
223              Tomb Raider           0.970940
253                  Deponia           0.969231
383            Anomaly Korea           0.969231
1142  Gothic Playable Teaser           0.959829
21             Shadowgrounds           0.957265


In [109]:
#Top 10 games by weighted degree
top_weighted = (
    results
    .sort_values(
        "weighted_degree",
        ascending=False
    )
    .head(10)
)

print(
    top_weighted[
        ["game", "weighted_degree"]
    ]
)

                        game  weighted_degree
162                Syberia 2           895555
10                    Portal           881600
161                  Syberia           876612
223              Tomb Raider           869094
283                 POSTAL 2           868985
277         Zombie Driver HD           868985
383            Anomaly Korea           867486
253                  Deponia           867486
1142  Gothic Playable Teaser           858844
21             Shadowgrounds           856379


In [110]:
#Top 10 games by betweenness centrality
top_betweenness = (
    results
    .sort_values(
        "betweenness_centrality",
        ascending=False
    )
    .head(10)
)

print(
    top_betweenness[
        ["game", "betweenness_centrality"]
    ]
)

                 game  betweenness_centrality
162         Syberia 2                0.448712
10             Portal                0.077201
161           Syberia                0.010381
283          POSTAL 2                0.007574
277  Zombie Driver HD                0.007574
253           Deponia                0.007022
383     Anomaly Korea                0.007022
223       Tomb Raider                0.006092
566            Millie                0.001802
21      Shadowgrounds                0.001628
